In [1]:
!pip install timm

Looking in indexes: http://mirrors.aliyun.com/pypi/simple


In [2]:
import os
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

In [3]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import timm
from torch.amp import autocast, GradScaler

# ==========================================
# 1. 全局配置与原论文超参数
# ==========================================
DATASET_NAME = "Pets"  # 可选: CIFAR10, CIFAR100, Pets, Flowers
NUM_CLASSES = 37
TOTAL_STEPS = 500        # Pets 数据集微调步数为 500 步 
BASE_LR = 0.01             # 论文建议的网格搜索范围: {0.001, 0.003, 0.01, 0.03} 
RESOLUTION = 384           # 微调阶段的输入分辨率统一放大为 384 [cite: 497, 507]

# A100 40GB 显存适配参数
LOGICAL_BATCH_SIZE = 512   # 论文规定的全局 Batch Size [cite: 497]
PHYSICAL_BATCH_SIZE = 512   # 物理 Batch Size
ACCUMULATION_STEPS = LOGICAL_BATCH_SIZE // PHYSICAL_BATCH_SIZE

# 评估频率
EVAL_EVERY_STEPS = 100     # 每 100 个逻辑步进行一次验证

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ==========================================
# 2. 数据处理与加载
# ==========================================
def get_dataloaders(batch_size):
    # Google ViT 官方归一化参数 [-1, 1]
    vit_mean, vit_std = (0.5, 0.5, 0.5), (0.5, 0.5, 0.5)
    resize_size = int(RESOLUTION / 0.875) # 约 438

    train_transform = transforms.Compose([
        transforms.Resize((resize_size, resize_size), interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.RandomCrop(RESOLUTION),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean=vit_mean, std=vit_std)
    ])

    eval_transform = transforms.Compose([
        transforms.Resize((resize_size, resize_size), interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.CenterCrop(RESOLUTION),
        transforms.ToTensor(),
        transforms.Normalize(mean=vit_mean, std=vit_std)
    ])

    print(f"Loading {DATASET_NAME} dataset...")
    train_dataset = torchvision.datasets.OxfordIIITPet(root='./autodl-tmp/data', split="trainval", download=False, transform=train_transform)
    test_dataset = torchvision.datasets.OxfordIIITPet(root='./autodl-tmp/data', split="test", download=False, transform=eval_transform)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=8, pin_memory=True, drop_last=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=8, pin_memory=True)
    
    return train_loader, test_loader

# ==========================================
# 3. 验证函数
# ==========================================
@torch.no_grad()
def evaluate(model, test_loader):
    model.eval()
    correct = 0
    total = 0
    
    for inputs, targets in test_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        # 验证阶段同样可以使用 BF16 加速
        with autocast(device_type='cuda', dtype=torch.bfloat16):
            outputs = model(inputs)
            
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()
        
    acc = 100. * correct / total
    model.train()
    return acc

# ==========================================
# 4. 主训练循环
# ==========================================
def main():
    train_loader, test_loader = get_dataloaders(PHYSICAL_BATCH_SIZE)

    # 加载预训练模型
    print("Loading ViT-B/32 pretrained model...")
    model = timm.create_model(
        'vit_base_patch32_224', 
        pretrained=True, 
        num_classes=NUM_CLASSES,
        img_size=RESOLUTION
    )

    # 论文强制要求：将新加入的分类头初始化为全零 [cite: 501]
    nn.init.zeros_(model.head.weight)
    nn.init.zeros_(model.head.bias)
    model = model.to(device)

    # 优化器：SGD，动量 0.9，无权重衰减 [cite: 489, 497]
    optimizer = optim.SGD(model.parameters(), lr=BASE_LR, momentum=0.9, weight_decay=0.0)
    
    # 调度器：全局 Cosine 退火 [cite: 497]
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=TOTAL_STEPS)
    
    criterion = nn.CrossEntropyLoss()
    scaler = GradScaler(enabled=True)

    print(f"Starting training for {TOTAL_STEPS} logical steps...")
    model.train()
    
    global_logical_step = 0
    optimizer.zero_grad()
    
    # 构造无限数据迭代器，因为论文是按 Step 训练而不是 Epoch
    def get_infinite_batches(dataloader):
        while True:
            for batch in dataloader:
                yield batch

    batch_iterator = get_infinite_batches(train_loader)
    
    for _ in range(TOTAL_STEPS * ACCUMULATION_STEPS):
        inputs, targets = next(batch_iterator)
        inputs, targets = inputs.to(device), targets.to(device)

        with autocast(device_type='cuda', dtype=torch.bfloat16):
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss = loss / ACCUMULATION_STEPS

        scaler.scale(loss).backward()

        # 达到物理 Batch 累加次数，执行真正的参数更新
        if (_ + 1) % ACCUMULATION_STEPS == 0:
            scaler.unscale_(optimizer)
            
            # 全局梯度裁剪，最大范数为 1.0 [cite: 497]
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()
            
            global_logical_step += 1

            # 打印日志
            if global_logical_step % 10 == 0:
                current_lr = scheduler.get_last_lr()[0]
                # 乘回累加步数以显示真实的全局 Batch Loss
                print(f"Step [{global_logical_step}/{TOTAL_STEPS}] | Loss: {loss.item() * ACCUMULATION_STEPS:.4f} | LR: {current_lr:.6f}")

            # 定期评估模型
            if global_logical_step % EVAL_EVERY_STEPS == 0 or global_logical_step == TOTAL_STEPS:
                acc = evaluate(model, test_loader)
                print(f"==> Evaluation at Step {global_logical_step}: Top-1 Accuracy = {acc:.2f}%")
                
            if global_logical_step >= TOTAL_STEPS:
                break

    print("Training Complete! Final evaluation running...")
    final_acc = evaluate(model, test_loader)
    print(f"Final Top-1 Accuracy on {DATASET_NAME}: {final_acc:.2f}%")

if __name__ == '__main__':
    main()

Loading Pets dataset...
Loading ViT-B/32 pretrained model...
Starting training for 500 logical steps...
Step [10/500] | Loss: 2.2651 | LR: 0.009990
Step [20/500] | Loss: 0.5909 | LR: 0.009961
Step [30/500] | Loss: 0.1959 | LR: 0.009911
Step [40/500] | Loss: 0.1391 | LR: 0.009843
Step [50/500] | Loss: 0.0842 | LR: 0.009755
Step [60/500] | Loss: 0.0559 | LR: 0.009649
Step [70/500] | Loss: 0.0366 | LR: 0.009524
Step [80/500] | Loss: 0.0329 | LR: 0.009382
Step [90/500] | Loss: 0.0277 | LR: 0.009222
Step [100/500] | Loss: 0.0199 | LR: 0.009045
==> Evaluation at Step 100: Top-1 Accuracy = 94.11%
Step [110/500] | Loss: 0.0204 | LR: 0.008853
Step [120/500] | Loss: 0.0143 | LR: 0.008645
Step [130/500] | Loss: 0.0148 | LR: 0.008423
Step [140/500] | Loss: 0.0136 | LR: 0.008187
Step [150/500] | Loss: 0.0104 | LR: 0.007939
Step [160/500] | Loss: 0.0135 | LR: 0.007679
Step [170/500] | Loss: 0.0093 | LR: 0.007409
Step [180/500] | Loss: 0.0098 | LR: 0.007129
Step [190/500] | Loss: 0.0094 | LR: 0.00684